In [1]:
from scipy.ndimage import zoom
import torch
import numpy as np

In [2]:
def resample_to_target_shape(data, target_shape):
    """
    Resample data to match the target shape
    """
    # Calculate zoom factors
    factors = (target_shape[0] / data.shape[0],
               target_shape[1] / data.shape[1],
               target_shape[2] / data.shape[2])

    # Resample using order=1 (linear interpolation) for continuous data
    resampled_data = zoom(data, factors, order=1)

    return resampled_data


In [3]:

def perform_region_occlusion_analysis(test_loader, model, device, atlas_data, region_labels, region_mapping, img_shape):
    """
    Perform occlusion analysis based on brain regions defined in the atlas
    """
    model.eval()

    print(f"Target image shape for resampling: {img_shape}")

    # Check if the atlas needs resampling
    if atlas_data.shape != img_shape:
        print(f"Resampling atlas from {atlas_data.shape} to {img_shape}")
        resampled_atlas = resample_to_target_shape(atlas_data, img_shape)
    else:
        print("Atlas already matches target shape, no resampling needed")
        resampled_atlas = atlas_data

    print(f"Resampled atlas shape: {resampled_atlas.shape}")

    # Initialize results dictionary to store effect per region
    region_occlusion_effects = {region: 0 for region in region_labels}
    region_sample_counts = {region: 0 for region in region_labels}

    print('======= Starting Region-Based Occlusion Analysis =============')

    sample_count = 0

    with torch.no_grad():
        for _, (input_img, ids, target, male) in enumerate(test_loader):
            # Print input_img shape for debugging
            print(f"Input image shape: {input_img.shape}")

            # Debugging: Print detailed shape information
            print(f"Input data type: {input_img.dtype}")

            # Get original prediction
            input_img = input_img.unsqueeze(0).to(device).type(torch.FloatTensor)

            # Handle gender information if needed by model
            # if opt.model == 'ScaleDense':
            #     male_onehot = torch.unsqueeze(male, 1)
            #     male_onehot = torch.zeros(male_onehot.shape[0], 2).scatter_(1, male_onehot, 1)
            #     male_onehot = male_onehot.type(torch.FloatTensor).to(device)
            #     original_output = model(input_img, male_onehot)
            # else:
            #     original_output = model(input_img)
            # print('input image shape:- ', input_img.shape)

            original_output = model(input_img)

            # original_output = original_output.cpu().numpy()
            original_output = original_output[0].numpy()

            # Process each region one by one
            for region in region_labels:
                # Free up memory
                torch.cuda.empty_cache()

                # Create mask for this region
                region_mask = (resampled_atlas == region)

                # Skip if region is not present in the resampled atlas
                if not np.any(region_mask):
                    continue

                # Clone the original input
                masked_input = input_img.clone()

                # Move input to CPU for masking
                # cpu_input = masked_input.cpu().numpy()

                cpu_input = masked_input.numpy()

                # Create a zero array with the same shape
                zeroed_array = np.zeros_like(cpu_input)

                # Create a mask array by broadcasting the region mask
                # This safely handles all dimension arrangements
                mask_array = np.ones_like(cpu_input)

                # Apply the region mask - this is the key change
                # We're assuming the last 3 dimensions of cpu_input correspond to the 3D volume
                for i in range(cpu_input.shape[0]):  # batch dimension
                    # Create a view that can be applied to the 3D volume regardless of channel arrangement
                    mask_view = np.broadcast_to(~region_mask, cpu_input[i].shape)
                    cpu_input[i] = cpu_input[i] * mask_view

                # Move back to GPU
                masked_input = torch.from_numpy(cpu_input).to(device)

                # Get prediction for masked input
                # if opt.model == 'ScaleDense':
                #     masked_output = model(masked_input, male_onehot)
                # else:
                #     masked_output = model(masked_input)

                masked_output = model(masked_input)

                # masked_output = masked_output.cpu().numpy()
                masked_output = masked_output[0].numpy()
                print("&&&"*40)
                print('region:- ',region_mapping[region])
                print('original_output:- ',original_output)
                print('masked_output:- ',masked_output)
                print("&&&"*40)

                # Calculate effect for this region (difference from original)
                effect = abs(masked_output - original_output)

                # Accumulate effect for this region
                region_occlusion_effects[region] += effect.item()
                region_sample_counts[region] += 1

                # Clean up
                del masked_input, cpu_input
                if 'masked_output' in locals():
                    del masked_output
                torch.cuda.empty_cache()

            sample_count += 1
            print(f"Processed sample {sample_count}/{len(test_loader)}: {ids[0]}")

    # Average effects across samples
    for region in region_labels:
        if region_sample_counts[region] > 0:
            region_occlusion_effects[region] /= region_sample_counts[region]

    # Convert results to a structured array
    result_array = np.array([region_occlusion_effects[region] for region in region_labels])

    return result_array



In [4]:
import os
import torch.nn as nn
import nibabel as nib
from nilearn import datasets

In [5]:
# ======== Load AAL atlas ======== #
aal_atlas = datasets.fetch_atlas_aal()
atlas_filename = aal_atlas.maps
atlas_nii = nib.load(atlas_filename)
atlas_data = atlas_nii.get_fdata()
region_labels = np.unique(atlas_data)[1:]  # Exclude 0 (background)
region_mapping = {code: label for code, label in zip(region_labels, aal_atlas.labels)}

print(f"Number of regions in atlas: {len(region_labels)}")
print(f"Atlas shape: {atlas_data.shape}")

C:\Users\Rishabh\AppData\Local\Temp\ipykernel_24704\3018833866.py:2: DeprecationWarning: Starting in version 0.13, the default fetched mask will beAAL 3v2 instead.
  aal_atlas = datasets.fetch_atlas_aal()


[fetch_atlas_aal] Dataset found in C:\Users\Rishabh\nilearn_data\aal_SPM12
Number of regions in atlas: 116
Atlas shape: (91, 109, 91)


In [6]:
import json
import os
import torch
# Load the configuration from the JSON file
with open(r'C:\Users\Rishabh\Documents\3d-hcct\config.json', 'r') as f:
    config = json.load(f)

In [7]:
from model import ViTForClassfication

# Initialize the model with the loaded configuration
model = ViTForClassfication(config=config)


In [8]:
from collections import OrderedDict
# checkpoint_path = r'C:\Users\Rishabh\training_output_metricsHCCT_best_model.pth.tar'
checkpoint_path = r'C:\Users\Rishabh\HCCT_checkpoint_withTransformation.pth.tar'

checkpoint = torch.load(checkpoint_path, map_location='cpu')
state_dict = checkpoint['state_dict']

# Remove 'module.' prefix if it exists
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k.replace('module.', '')  # strip the prefix
    new_state_dict[name] = v

model.load_state_dict(new_state_dict, strict=True)  # strict ensures all match



<All keys matched successfully>

In [9]:
import os
import torch
import nibabel as nib
import numpy as np
import pandas as pd
import torch.nn.functional as F


def nii_loader(path, dtype=np.float32, mmap_mode='r'):
    """
    Load NIfTI file with memory mapping option for large files.

    Args:
        path: Path to NIfTI file
        dtype: Data type to cast to (default: float32)
        mmap_mode: Memory mapping mode (default: 'r' for read-only)
                   Set to None to load data into memory
    """
    img = nib.load(str(path))
    # Use memory mapping for large files
    data = img.get_fdata(dtype=dtype, caching='unchanged')
    return data


def read_table(path):
    """Read Excel table and return values"""
    return pd.read_excel(path, header=None).values


def white0(image, threshold=0):
    """
    Standardize voxels with value > threshold

    Args:
        image: Input image
        threshold: Threshold value

    Returns:
        Standardized image
    """
    image = image.astype(np.float32)
    mask = (image > threshold).astype(int)

    # Vectorized implementation to avoid unnecessary memory allocation
    image_h = image * mask

    # Calculate mean and std only for relevant voxels
    non_zero_voxels = np.sum(mask)
    if non_zero_voxels > 0:
        mean = np.sum(image_h) / non_zero_voxels

        # More memory efficient way to calculate std
        std_sum = np.sum((image_h - mean * mask) ** 2)
        std = np.sqrt(std_sum / non_zero_voxels)

        if std > 0:
            normalized = mask * (image - mean) / std
            # Use in-place operations to reduce memory usage
            image = normalized + image * (1 - mask)
            return image

    # Default case
    return np.zeros_like(image, dtype=np.float32)


class IMG_Folder(torch.utils.data.Dataset):
    """
    Dataset class for loading brain images with memory optimizations
    """

    def __init__(self, excel_path, data_path, loader=nii_loader, transforms=None, preload=False):
        """
        Args:
            excel_path: Path to Excel file with metadata
            data_path: Path to directory with NIfTI files
            loader: Function to load NIfTI files
            transforms: Transforms to apply to images
            preload: Whether to preload all data into memory (default: False)
        """
        self.root = data_path
        self.sub_fns = sorted(os.listdir(self.root))
        self.table_refer = read_table(excel_path)
        self.loader = loader
        self.transform = transforms
        self.preload = preload

        # Create a mapping from subject ID to metadata for faster lookup
        self.metadata = {}
        for f in self.table_refer:
            # print('f:- ',f)
            # print('f[1]:- ', f[1])
            sid = str(f[0])
            # slabel = int(f[1])
            slabel = f[1]
            smale = f[2]
            self.metadata[sid] = (slabel, smale)

        # Optionally preload all data into memory
        if preload:
            self.cached_data = {}
            for sub_fn in self.sub_fns:
                if sub_fn in self.metadata:
                    sub_path = os.path.join(self.root, sub_fn)
                    self.cached_data[sub_fn] = self.loader(sub_path)

    def __len__(self):
        return len(self.sub_fns)

    def __getitem__(self, index):
        sub_fn = self.sub_fns[index]

        # Get metadata for this subject
        if sub_fn not in self.metadata:
            # Find manually if not in mapping (fallback)
            for f in self.table_refer:
                sid = str(f[0])
                # slabel = int(f[1])
                slabel = f[1]
                smale = f[2]
                if sid == sub_fn:
                    break
        else:
            slabel, smale = self.metadata[sub_fn]
            sid = sub_fn

        # Load image data
        if self.preload and sub_fn in self.cached_data:
            img = self.cached_data[sub_fn].copy()  # Make a copy to avoid modifying cached data
        else:
            sub_path = os.path.join(self.root, sub_fn)
            img = self.loader(sub_path)

        # Preprocessing
        img = white0(img)

        # Apply transforms if provided
        if self.transform is not None:
            img = self.transform(img)

        # Convert to contiguous float tensor
        img = np.ascontiguousarray(img, dtype=np.float32)
        img = torch.from_numpy(img).type(torch.FloatTensor)

        return (img, sid, slabel, smale)

In [10]:
CheckpointPath = r'C:\Users\Rishabh\trainingMulti_VIT_best_model.pth.tar'
CSVPath = r'C:\Users\Rishabh\Documents\TransBTS\IXI.xlsx'
DataFolder = r'C:\Users\Rishabh\Documents\TrimeseData'
test_data = IMG_Folder(CSVPath, DataFolder)
device = "cpu"

In [11]:
valid_loader = torch.utils.data.DataLoader(test_data
                                         ,batch_size=1
                                         ,num_workers=0
                                         ,pin_memory=True
                                         ,drop_last=True
                                         )

In [12]:
# Get sample image to determine exact shape
for sample_data in valid_loader:
    sample_img = sample_data[0]
    img_shape = (sample_img.shape[1], sample_img.shape[2], sample_img.shape[3])
    print(f"Detected image shape: {img_shape}")
    break

Detected image shape: (91, 109, 91)


C:\Users\Rishabh\anaconda3\envs\3d-hcct\Lib\site-packages\torch\cuda\__init__.py:182: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0
C:\Users\Rishabh\anaconda3\envs\3d-hcct\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [13]:
# ======== perform region-based occlusion analysis ======== #
region_occlusion_results = perform_region_occlusion_analysis(
    test_loader=valid_loader,
    model=model,
    device="cpu",
    atlas_data=atlas_data,
    region_labels=region_labels,
    region_mapping=region_mapping,
    img_shape=img_shape
)


Target image shape for resampling: (91, 109, 91)
Atlas already matches target shape, no resampling needed
Resampled atlas shape: (91, 109, 91)
======= Starting Region-Based Occlusion Analysis =============
Input image shape: torch.Size([1, 91, 109, 91])
Input data type: torch.float32
&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&
region:-  Background
original_output:-  [25.702253]
masked_output:-  [23.811289]
&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&
&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&
region:-  Precentral_L
original_output:-  [25.702253]
masked_output:-  [22.715233]
&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&
&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&